### MiniZinc

[Video](https://youtu.be/6AqdzJ0KGQk)

MiniZinc ist eine Modellierungssprache für Constraint-Programmierung (CP), die verwendet wird, um Optimierungs- und Entscheidungsprobleme zu formulieren. Man beschreibt das Problem in MiniZinc, und ein sogenannter Solver findet dann eine Lösung.

Typische Problemstellungen, die mit MiniZinc gelöst werden können, sind solche, bei denen man eine Menge von Bedingungen (Constraints) erfüllen und eventuell ein Optimierungskriterium verbessern möchte.

Die folgenden Beispiele stammen aus einem Kurs der Chinese University of Hong Kong. Viele davon haben alle eine etwas kriegerische Note.


#### Four Villages

Es soll eine Armee aus Soldaten aus 4 Dörfern zusammengestellt werden. Sie kosten unterschiedlich und haben unterschiedliche Stärke.
Der Einkauf ist durch ein Budget begrenzt

<img src='army.png' width='400'>

Zwei Arten von Variablen:

    Parameter: wie Variablen in Programmiersprachen, mit Typ.
    Decision Variables: Ausdrücke mit decision variables folgen stärkeren Regeln als für Parameter.

In [1]:
solve maximize 6*F + 10*L + 8*Z + 40*J;             % objective

int: budget = 10000;                                % Parameter 

constraint 13*F + 21*L + 17*Z + 100*J <= budget;    % constraints

var 0..1000: F;                                     % Decision Variables  
var 0..400: L;
var 0..500: Z;
var 0..150: J;

output ["F = \(F), L = \(L), Z = \(Z), J = \(J)\n"]; % Output

SyntaxError: invalid syntax (846675125.py, line 1)

In [ ]:
# optimales Ergebnis
F = 0, L = 392, Z = 104, J = 0

#### Counting Soldiers

Wir wissen die Reste, wenn wir die soldiers in Reihen zu 5, 7 oder 12 aufstellen. Das Problem ist ohne *objective*, es ist ein
Satisfaction Problem, da wird standardmäßig nur nach *einer* Lösung gesucht. Man kann in der IDE einstellen, dass man alle Lösungen will.

In [ ]:
var 100..800: army;

constraint army mod 5 = 2;
constraint army mod 7 = 2;
constraint army mod 12 = 1;

solve satisfy;

In [ ]:
# Ergebnis  (bei allen Ergebnissen würde der Doppelstrich erscheinen)
army = 457;
----------

Die Antwort lässt sich mathematisch mit dem Chinesischen Restsatz berechnen.

#### Colouring a Map

<img src='vierFarben.png' width='401'>

In [ ]:
enum COLOR = {GREEN, BLUE, PINK, YELLOW};
    
var COLOR: Si;
var COLOR: Yan; 
var COLOR: Yu;
var COLOR: Xu; 
var COLOR: Qing;
var COLOR: Ji; 
var COLOR: You;
var COLOR: Bing; 
var COLOR: Yong;
var COLOR: Liang; 
var COLOR: Yi;
var COLOR: Jing; 
var COLOR: Yang;
var COLOR: Jiao;

constraint Liang != Yong;
constraint Yong != Yi;
constraint Yong != Jing;
constraint Yong != Si;
constraint Yi != Jing;
constraint Yi != Jiao;
constraint Jiao != Jing;
constraint Jiao != Yang;
constraint Jing != Yang;
constraint Jing != Yong;
constraint Jing != Si;
constraint Jing != Yu;
constraint Yang != Yu;
constraint Yang != Xu;
constraint Yu != Si;
constraint Yu != Yan;
constraint Yu != Xu;
constraint Xu != Yan;
constraint Xu != Qing;
constraint Yan != Si;
constraint Yan != Ji;
constraint Yan != Ji;
constraint Yan != Qing;
constraint Qing != Ji;
constraint Ji != You;
constraint Ji != Bing;
constraint Ji != Si;
constraint You != Bing;
constraint Bing != Si;

solve satisfy;

In [ ]:
# Ergebnis
Si = PINK;
Yan = YELLOW;
Yu = BLUE;
Xu = GREEN;
Qing = PINK;
Ji = BLUE;
You = PINK;
Bing = YELLOW;
Yong = BLUE;
Liang = YELLOW;
Yi = PINK;
Jing = YELLOW;
Yang = PINK;
Jiao = BLUE;
----------
==========

#### Banquet

Verschiedene Gerichte haben einen Wohlschmeckfaktor, brauchen aber Platz am Tisch. Der Tisch hat eine beschränkte Länge.

<img src='banquet.png' width='250'>

In [ ]:
enum DISH;
int: capacity;
array[DISH] of int: satisf;
array[DISH] of int: size;

array[DISH] of var int: amt; % how many of each dish

constraint forall(i in DISH)(amt[i] >= 0);
constraint sum(i in DISH)(size[i] * amt[i]) <= capacity;
solve maximize sum(i in DISH)(satisf[i] * amt[i]);

output ["Amount = ", show(amt), "\n"];

In [ ]:
% dzn
enum DISH = {SNAKESOUP, GONGBAOFROGS, MAPOTOFU};
capacity = 18;
satisf = [29,19,8];
size = [8,5,3];

In [ ]:
# Ergebnis
----------
Amount = [1, 2, 0]
----------
==========


#### Weapons

Es sollen Waffen produziert werden. Jede Waffe hat eine Stärke und für die Produktion werden Anteile von 4 Ressourcen benötigt. Die Ressourcen sind begrenzt. Wieviele Waffen sollen je Typ produziert werden für eine maximale Stärke?

<img src='weapons1.png' width='400'>

<img src='weapons2.png' width='400'>

<img src='weapons3.png' width='400'>

Die Information, wieviel Resourcen pro Waffe verbraucht werden, werden in einer Matrix gespeichert.

In [ ]:
enum WEAPON = {AXE, SWORD, PIKE, SPEAR, CLUB};
enum RESOURCE = {IRON, WOOD, SMITH, CARPENTER};

array[WEAPON] of float: strength = [11.0, 18.0, 15.0, 17.0, 11.0];
array[RESOURCE] of float: capacity = [5000, 7500, 4000, 3000]; 
array[WEAPON,RESOURCE] of float: consumption = [| 1.5, 1.0, 1.0, 1.0 
                                                | 2.0, 0.0, 2.0, 0.0
                                                | 1.5, 0.5, 1.0, 1.0 
                                                | 0.5, 1.0, 0.9, 1.5
                                                | 0.1, 2.5, 0.1, 2.5 |];
% decision variables
array[WEAPON] of var int: amt;       
 
% constraints
constraint forall(w in WEAPON) (amt[w] >= 0);
constraint forall (r in RESOURCE)(
      sum (w in WEAPON)(consumption[w, r] * amt[w]) <= capacity[r]
); 

% objective
solve maximize sum(w in WEAPON)(strength[w]*amt[w]);

% output
output [ "\(w): \(amt[w])\n" | w in WEAPON ]++["Strength: \(sum(w in WEAPON)(strength[w]*amt[w]))"];

#### Counting Rods

Für 5 Zahlen sind die rods runtergefallen.  

<img src='rods1.png' width='400'>

<img src='rods2.png' width='400'>

<img src='rods3.png' width='400'>

Wir können auch eine Decision-Variable nutzen um in Lookup im Array zu machen.

In [ ]:
set of int: DIGIT = 1..9;
array[DIGIT] of int: rods = [1,2,3,4,5,2,3,4,5];  

var DIGIT: M1; % first messed up digit
var DIGIT: M2; % second messed up digit
var DIGIT: M3; % third messed up digit
var DIGIT: M4; % fourth messed up digit
var DIGIT: M5; % fifth messed up digit


constraint rods[M1] + rods[M2] + rods[M3] +  
           rods[M4] + rods[M5] = 12;

constraint 2303 + M1 * 10 + 980 + M2 * 1000 + M3 
           = 301 + M4 * 1000 + M5 * 10; 

% alldifferent messed up digits
include "alldifferent.mzn";

constraint alldifferent([M1,M2,M3,M4,M5]);

solve satisfy;